# 3W Dataset — Starter Notebook: Differential & Spectral Feature Builder

This notebook is a **starting point** for the feature layer described in the methodology
document. It builds the physics-informed features from a 3W parquet instance:

- **Differentials**: `dP_CKP`, `dP_CKGL`, `dT_CKP` (Joule–Thomson cooling),
  `dP_grad = P-PDG − P-TPT`, plus a normalised choke coefficient `Cv_eff`.
- **Spectral / oscillation** descriptors for the steady-state faults
  (severe slugging, flow instability): dominant frequency, spectral entropy, band-power.
- **Quality channels**: frozen-sensor and missing-value indicators.
- **Within-instance normalisation** so models learn physics, not well-specific baselines.

It runs **out of the box** on a synthetic instance if no real parquet files are found,
so you can execute top-to-bottom immediately, then point `DATA_DIR` at the real
3W parquet files to use it for real.

> Scope note: this is exploratory scaffolding, not the production `features` module.
> Promote the functions here into `src/threew/features/` with tests (see `AGENT.md`).

In [ ]:
# --- dependencies (already present in most DS environments) ---
# If needed: pip install pandas numpy pyarrow scipy matplotlib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import signal
import matplotlib.pyplot as plt

np.random.seed(7)
pd.set_option("display.max_columns", 40)
print("imports ok")

## 1. Configuration

In [ ]:
# Point this at the directory holding 3W .parquet instances.
# If it does not exist (or is empty), the notebook falls back to a synthetic instance.
DATA_DIR = Path("/mnt/user-data/uploads")     # <-- change to your 3W parquet folder
RESAMPLE_RATE = "1s"                            # regular grid for time alignment
ROLL_WINDOW   = 64                              # samples; rolling stats / freezing
SPECTRAL_WINDOW = 256                           # samples; FFT window for oscillation features
EPS = 1e-9                                       # division guard

# Canonical 3W signal columns we rely on here (subset of the full 29).
CHANNELS = [
    "ABER-CKGL", "ABER-CKP",
    "P-ANULAR", "P-JUS-CKGL", "P-JUS-CKP", "P-MON-CKGL", "P-MON-CKP",
    "P-MON-SDV-P", "P-PDG", "PT-P", "P-TPT", "P-JUS-BS",
    "QBS", "QGL",
    "T-JUS-CKP", "T-MON-CKP", "T-PDG", "T-TPT",
    "ESTADO-DHSV",
]
LABEL_COLS = ["class", "state"]

## 2. Load a parquet instance (with synthetic fallback)

In [ ]:
def make_synthetic_instance(n=3000, fault="SEVERE_SLUGGING"):
    '''Generate a physically-plausible synthetic 3W-like instance for demo purposes.
    Includes a slugging-style oscillation in the second half so spectral features fire.'''
    t = pd.date_range("2024-01-01", periods=n, freq="s")
    base_p = 2.0e7   # ~200 bar in Pa
    df = pd.DataFrame(index=t)

    # upstream/downstream pressures around the production choke
    df["P-MON-CKP"] = base_p + np.random.normal(0, 2e4, n)
    dp = 6.0e5 + np.random.normal(0, 1e4, n)              # ~6 bar drop
    df["P-JUS-CKP"] = df["P-MON-CKP"] - dp

    # gas-lift choke
    df["P-MON-CKGL"] = 1.2e7 + np.random.normal(0, 1e4, n)
    df["P-JUS-CKGL"] = df["P-MON-CKGL"] - (3.0e5 + np.random.normal(0, 8e3, n))

    # downhole vs tree
    df["P-PDG"] = 2.5e7 + np.random.normal(0, 3e4, n)
    df["P-TPT"] = 1.9e7 + np.random.normal(0, 3e4, n)
    df["PT-P"]  = df["P-TPT"] - 1e5
    df["P-ANULAR"] = 1.1e7 + np.random.normal(0, 1e4, n)
    df["P-MON-SDV-P"] = df["P-JUS-CKP"] + 5e4
    df["P-JUS-BS"] = 9e6 + np.random.normal(0, 1e4, n)

    # temperatures
    df["T-MON-CKP"] = 60 + np.random.normal(0, 0.3, n)
    df["T-JUS-CKP"] = df["T-MON-CKP"] - (8 + np.random.normal(0, 0.2, n))  # JT cooling
    df["T-PDG"] = 90 + np.random.normal(0, 0.2, n)
    df["T-TPT"] = 45 + np.random.normal(0, 0.3, n)

    # flows & openings
    df["QGL"] = 0.05 + np.random.normal(0, 1e-3, n)
    df["QBS"] = 0.01 + np.random.normal(0, 5e-4, n)
    df["ABER-CKP"]  = 70.0 + np.random.normal(0, 0.5, n)
    df["ABER-CKGL"] = 55.0 + np.random.normal(0, 0.5, n)
    df["ESTADO-DHSV"] = 1.0

    # inject a slugging-style oscillation into the downhole/tree pressures (2nd half)
    half = n // 2
    osc = 4e5 * np.sin(2 * np.pi * 0.02 * np.arange(n - half))   # ~0.02 Hz slug cycle
    df.iloc[half:, df.columns.get_loc("P-PDG")] += osc
    df.iloc[half:, df.columns.get_loc("P-TPT")] += 0.8 * osc

    # labels: NORMAL (0) then SEVERE_SLUGGING (3) in the oscillatory half
    cls = np.zeros(n, dtype=int); cls[half:] = 3
    df["class"] = cls
    df["state"] = np.where(cls == 0, "normal", "abnormal")
    df.index.name = "timestamp"
    return df

def load_first_instance(data_dir: Path):
    files = sorted(data_dir.glob("*.parquet")) if data_dir.exists() else []
    if files:
        print(f"Loading real instance: {files[0].name}")
        df = pd.read_parquet(files[0], engine="pyarrow")
        # standardise the time index
        ts_col = "timestamp" if "timestamp" in df.columns else df.columns[0]
        df[ts_col] = pd.to_datetime(df[ts_col])
        df = df.set_index(ts_col).sort_index()
        return df, False
    print("No parquet found in DATA_DIR — using a synthetic demo instance.")
    return make_synthetic_instance(), True

raw, is_synth = load_first_instance(DATA_DIR)
print("shape:", raw.shape)
raw.head()

## 3. Resample to a regular grid + quality channels

Real 3W data has irregular sampling, frozen sensors and gaps. We resample to a regular grid, carry-forward valve states, and emit explicit `is_missing` / `is_frozen` indicators instead of silently imputing.

In [ ]:
def add_quality_and_resample(df, rate=RESAMPLE_RATE, roll=ROLL_WINDOW):
    present = [c for c in CHANNELS if c in df.columns]
    g = df[present + [c for c in LABEL_COLS if c in df.columns]].copy()

    # regular grid
    g = g.resample(rate).first()

    # missingness BEFORE filling
    miss = g[present].isna().astype(int).add_suffix("__is_missing")

    # carry-forward fill (valve states must never be interpolated)
    g[present] = g[present].ffill().bfill()

    # frozen-sensor flag: near-zero rolling std
    roll_std = g[present].rolling(roll, min_periods=roll // 2).std()
    frozen = (roll_std < 1e-6).astype(int).add_suffix("__is_frozen")

    return pd.concat([g, miss, frozen], axis=1)

clean = add_quality_and_resample(raw)
print("after resample:", clean.shape)
clean.filter(regex="is_missing|is_frozen").sum().sort_values(ascending=False).head(8)

## 4. Differential features

The core physics: cross-component ΔP/ΔT and a normalised choke coefficient. These are far more transferable across wells than absolute readings.

In [ ]:
def build_differentials(df):
    out = pd.DataFrame(index=df.index)

    def diff(a, b, name):
        if a in df and b in df:
            out[name] = df[a] - df[b]

    # pressure drops across chokes
    diff("P-MON-CKP",  "P-JUS-CKP",  "dP_CKP")     # production choke ΔP -> restriction/scaling
    diff("P-MON-CKGL", "P-JUS-CKGL", "dP_CKGL")    # gas-lift choke ΔP
    # Joule-Thomson cooling across the production choke -> hydrate precursor
    diff("T-MON-CKP",  "T-JUS-CKP",  "dT_CKP")
    # downhole-to-tree gradients -> fluid density / BSW, productivity
    diff("P-PDG", "P-TPT", "dP_grad")
    diff("T-PDG", "T-TPT", "dT_grad")

    # normalised choke coefficient: Q / (opening * sqrt(dP)) — near-invariant under normal flow
    if {"QGL", "ABER-CKGL"}.issubset(df.columns) and "dP_CKGL" in out:
        denom = (df["ABER-CKGL"].clip(lower=EPS) *
                 np.sqrt(out["dP_CKGL"].clip(lower=EPS)))
        out["Cv_eff_GL"] = df["QGL"] / (denom + EPS)
    if {"QBS", "ABER-CKP"}.issubset(df.columns) and "dP_CKP" in out:
        denom = (df["ABER-CKP"].clip(lower=EPS) *
                 np.sqrt(out["dP_CKP"].clip(lower=EPS)))
        out["Cv_eff_P"] = df["QBS"] / (denom + EPS)

    return out

diffs = build_differentials(clean)
print("differential features:", list(diffs.columns))
diffs.describe().T[["mean", "std", "min", "max"]]

## 5. Spectral & oscillation features

Severe slugging (class 3) and flow instability (class 4) are **sustained oscillatory regimes** invisible to statistical moments. We compute rolling spectral descriptors on the pressure channels that carry the oscillation.

In [ ]:
def spectral_entropy(x):
    x = np.asarray(x, float)
    x = x - x.mean()
    f, pxx = signal.periodogram(x)
    pxx = pxx[1:]                      # drop DC
    if pxx.sum() <= 0:
        return 0.0, 0.0, 0.0
    p = pxx / pxx.sum()
    ent = -np.sum(p * np.log(p + EPS)) / np.log(len(p) + EPS)   # normalised [0,1]
    dom_f = f[1:][np.argmax(pxx)]                                # dominant frequency
    band = pxx[(f[1:] > 0.005) & (f[1:] < 0.05)].sum() / pxx.sum()  # slug band power
    return float(dom_f), float(ent), float(band)

def build_spectral(df, cols=("P-PDG", "P-TPT", "PT-P", "QGL"),
                   win=SPECTRAL_WINDOW, step=None):
    step = step or win // 4
    cols = [c for c in cols if c in df.columns]
    rows = []
    idx = []
    for start in range(0, len(df) - win, step):
        seg = df.iloc[start:start + win]
        rec = {}
        for c in cols:
            dom, ent, band = spectral_entropy(seg[c].values)
            rec[f"{c}__dom_f"]   = dom
            rec[f"{c}__spec_ent"] = ent
            rec[f"{c}__slug_band"] = band
            # oscillation amplitude of the detrended segment
            rec[f"{c}__osc_amp"] = float(signal.detrend(seg[c].values).std())
        rows.append(rec)
        idx.append(df.index[start + win // 2])
    return pd.DataFrame(rows, index=pd.Index(idx, name="timestamp"))

spec = build_spectral(clean)
print("spectral windows:", spec.shape)
spec.filter(regex="P-PDG").head()

## 6. Within-instance normalisation

Normalise continuous features using statistics from the **normal** segment only, so the model sees deviations from this well's own baseline rather than absolute levels.

In [ ]:
def normalise_within_instance(df, label=None):
    out = df.copy()
    if label is not None and (label == 0).any():
        ref = df[label == 0]               # use NORMAL rows as the reference
    else:
        ref = df
    mu, sd = ref.mean(), ref.std().replace(0, 1.0)
    return (out - mu) / sd

# align the per-sample class label onto the differential frame, then normalise
cls = clean["class"] if "class" in clean else None
diffs_norm = normalise_within_instance(diffs, cls)
diffs_norm.describe().T[["mean", "std"]].head()

## 7. Quick visual sanity check

The `dP_grad` and the slug-band power should both react in the oscillatory (fault) half of the demo instance.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=False)

ax[0].plot(clean.index, clean["P-PDG"], lw=0.7)
ax[0].set_title("P-PDG (downhole pressure) — oscillation appears in the fault half")
ax[0].set_ylabel("Pa")

if "dP_grad" in diffs:
    ax[1].plot(diffs.index, diffs["dP_grad"], lw=0.7, color="tab:purple")
    ax[1].set_title("dP_grad = P-PDG − P-TPT (density / regime indicator)")
    ax[1].set_ylabel("Pa")

if "P-PDG__slug_band" in spec:
    ax[2].plot(spec.index, spec["P-PDG__slug_band"], marker=".", color="tab:red")
    ax[2].set_title("P-PDG slug-band power (rises during slugging)")
    ax[2].set_ylabel("frac")

plt.tight_layout(); plt.show()

## 8. Assemble the feature matrix

Merge the per-sample differentials with the windowed spectral features (as-of join) to produce one feature row per spectral window — the unit the baseline GBT model will consume.

In [ ]:
def assemble(diffs_norm, spec, clean):
    feat = pd.merge_asof(
        spec.sort_index(),
        diffs_norm.sort_index(),
        left_index=True, right_index=True, direction="nearest",
    )
    # attach quality summary (mean over the nearest sample) + label
    qual = clean.filter(regex="is_missing|is_frozen")
    feat = pd.merge_asof(feat, qual.sort_index(), left_index=True,
                         right_index=True, direction="nearest")
    if "class" in clean:
        feat = pd.merge_asof(feat, clean[["class"]].sort_index(),
                             left_index=True, right_index=True, direction="nearest")
    return feat

features = assemble(diffs_norm, spec, clean)
print("final feature matrix:", features.shape)
print("label distribution:\n", features.get("class", pd.Series(dtype=int)).value_counts())
features.head()

## 9. Where to go next

1. **Promote** these functions into `src/threew/features/` as pure, tested functions
   (see `AGENT.md` §5.2) — including the `Cv_eff` division-guard and frozen-sensor edge cases.
2. **Window the labels** with the transient (`+100`) offset preserved, so the transient head
   has targets.
3. **Build the baseline**: LightGBM/XGBoost over this matrix on a **well-disjoint** split.
4. **Iterate** toward the spatio-temporal MoE only once the feature set beats a naive baseline.

> Reminder: never evaluate without well-disjoint splits, and always report
> time-to-detection alongside macro-F1.